# E04 — Continual Learning Extension and Unified Comparison

This notebook extends the group continual-learning experiment without creating a second task map, dataset implementation, scratch no-replay trainer, or scratch replay trainer.

It uses the current `main` branch infrastructure:

- `data/processed/continual_100/class_tasks_100.csv`
- `src.config.CONTINUAL_NUM_CLASSES`
- `src.config.CONTINUAL_CLASSES_PER_TASK`
- `src.config.CONTINUAL_CLASS_TASKS_CSV`
- `src.data.create_continual_dataset`
- `src.data.create_dataloader`
- `src.advanced.continual_metrics`
- `src.advanced.continual_replay`
- `scripts/train_continual_no_replay.py`
- `scripts/train_continual_replay.py`
- `src.evaluation.evaluate_class_scores`

The notebook adds only the following E04 components:

1. a scratch joint-training upper bound;
2. a pretrained joint-training upper bound;
3. a unified pretrained sequential trainer supporting no replay or class-balanced replay;
4. optional final 100-class test evaluation after validation-based configuration selection;
5. optional representation-drift analysis, disabled by default;
6. an automatic Step 13 summariser that reads run directories directly.

**Important:** the held-out test split must not be used to select learning rate, memory budget, epoch count, or any other hyperparameter.

## Step 1 — Import dependencies and locate the project root

In [ ]:
import csv
import hashlib
import json
import random
import subprocess
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Subset
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the project root. Run this notebook from inside "
        "9517_assignment_MVP_Group."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.advanced.continual_metrics import (
    create_accuracy_matrix,
    record_accuracy_row,
    summarize_after_task,
)
from src.advanced.continual_replay import (
    update_class_balanced_memory,
    validate_class_balanced_memory,
)
from src.data import create_continual_dataset, create_dataloader
from src.deep_learning import (
    create_scratch_resnet18,
    train_one_epoch,
    validate_one_epoch,
)
from src.evaluation import evaluate_class_scores

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


## Step 2 — Validate the shared 100-class continual-learning plan

This cell deliberately reads only the committed shared task plan. It does not generate or use `data/processed/cl_tasks.csv`.

In [ ]:
NUM_CLASSES = config.CONTINUAL_NUM_CLASSES
CLASSES_PER_TASK = config.CONTINUAL_CLASSES_PER_TASK

if NUM_CLASSES != 100:
    raise ValueError(f"Expected CONTINUAL_NUM_CLASSES=100, found {NUM_CLASSES}")
if CLASSES_PER_TASK != 10:
    raise ValueError(
        f"Expected CONTINUAL_CLASSES_PER_TASK=10, found {CLASSES_PER_TASK}"
    )
if NUM_CLASSES % CLASSES_PER_TASK != 0:
    raise ValueError("The continual class count must be divisible by classes per task")

NUM_TASKS = NUM_CLASSES // CLASSES_PER_TASK
TASK_IDS = list(range(NUM_TASKS))
TASK_PLAN_PATH = Path(config.CONTINUAL_CLASS_TASKS_CSV)

if not TASK_PLAN_PATH.is_file():
    raise FileNotFoundError(f"Shared task plan not found: {TASK_PLAN_PATH}")

task_plan = pd.read_csv(TASK_PLAN_PATH)
required_columns = {
    "task_id",
    "task_label",
    "continual_label",
    "source_label",
    "category_id",
    "category_name",
}
missing_columns = required_columns.difference(task_plan.columns)
if missing_columns:
    raise ValueError(
        "Shared task plan is missing columns: "
        + ", ".join(sorted(missing_columns))
    )

if len(task_plan) != NUM_CLASSES:
    raise ValueError(
        f"Expected {NUM_CLASSES} task-plan rows, found {len(task_plan)}"
    )

expected_continual_labels = list(range(NUM_CLASSES))
actual_continual_labels = sorted(task_plan["continual_label"].astype(int).tolist())
if actual_continual_labels != expected_continual_labels:
    raise ValueError("The shared continual labels are not exactly 0–99")

task_counts = task_plan.groupby("task_id").size().sort_index()
if task_counts.to_dict() != {task_id: CLASSES_PER_TASK for task_id in TASK_IDS}:
    raise ValueError("Each continual task must contain exactly 10 classes")

forbidden_task_map = PROJECT_ROOT / "data" / "processed" / "cl_tasks.csv"
if forbidden_task_map.exists():
    warnings.warn(
        f"Ignoring non-shared task map: {forbidden_task_map}. "
        "This notebook uses only config.CONTINUAL_CLASS_TASKS_CSV."
    )

TASK_PLAN_SHA256 = hashlib.sha256(TASK_PLAN_PATH.read_bytes()).hexdigest()

print(f"Tasks: {NUM_TASKS}")
print(f"Classes per task: {CLASSES_PER_TASK}")
print(f"Shared task plan: {TASK_PLAN_PATH}")
print(f"Task-plan SHA-256: {TASK_PLAN_SHA256}")
display(task_plan.head(12))


## Step 3 — Experiment switches and fixed validation-selected settings

All expensive runs are disabled by default. Enable only the run that you intend to execute.

The default epoch counts are compute-matched:

- sequential exposure: `10 tasks × 20 epochs × 400 current-task images`;
- joint exposure: `20 epochs × 4,000 images`.

Both equal 80,000 current-data image presentations before replay samples are added.

In [ ]:
# Existing D scratch entry points.
RUN_SCRATCH_NO_REPLAY = True
RUN_SCRATCH_REPLAY_M2 = True
RUN_SCRATCH_REPLAY_M5 = True

# E04 additions.
RUN_SCRATCH_JOINT = True
RUN_PRETRAINED_JOINT = True
RUN_PRETRAINED_NO_REPLAY = True
RUN_PRETRAINED_REPLAY_M2 = True
RUN_PRETRAINED_REPLAY_M5 = True

# Run only after all hyperparameters and the selected method are fixed on validation.
RUN_FINAL_TEST = True

# Optional analysis; it uses an existing trained pretrained checkpoint.
RUN_DRIFT_ANALYSIS = False

# Safe to leave enabled. Missing run directories are skipped.
RUN_STEP13_SUMMARY = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
NUM_WORKERS = 4 if torch.cuda.is_available() else 0
EPOCHS_PER_TASK = 20
JOINT_EPOCHS = 20

SCRATCH_LEARNING_RATE = 1e-2
PRETRAINED_LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
MOMENTUM = 0.9
PRETRAINED_LABEL_SMOOTHING = 0.1

TRAIN_AUGMENTATION = True
RESUME_EXISTING_RUNS = True

OUTPUT_ROOT = Path(config.OUTPUT_ROOT) / "continual_100"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_DIRECTORIES = {
    "scratch_no_replay": OUTPUT_ROOT / "no_replay_v1",
    "scratch_replay_m2": OUTPUT_ROOT / "replay_m2_v1",
    "scratch_replay_m5": OUTPUT_ROOT / "replay_m5_v1",
    "scratch_joint": OUTPUT_ROOT / "joint_v1",
    "pretrained_joint": OUTPUT_ROOT / "pretrained_joint_v1",
    "pretrained_no_replay": OUTPUT_ROOT / "pretrained_no_replay_v1",
    "pretrained_replay_m2": OUTPUT_ROOT / "pretrained_replay_m2_v1",
    "pretrained_replay_m5": OUTPUT_ROOT / "pretrained_replay_m5_v1",
}

# Fill this only after validation-based method selection is complete.
# Example: TEST_RUN_KEYS = ["scratch_replay_m5", "pretrained_replay_m5"]
TEST_RUN_KEYS = ["pretrained_replay_m5",
                ]

# Representation drift is intended only for a pretrained run.
DRIFT_RUN_KEY = "pretrained_no_replay"
DRIFT_MAX_SAMPLES = 256

print(f"Device: {DEVICE}")
for run_key, run_dir in RUN_DIRECTORIES.items():
    print(f"{run_key:26s} -> {run_dir}")


## Step 4 — Reproducibility, transforms, loaders, and artifact helpers

In [ ]:
def set_seed(seed: int = config.RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def create_train_transform():
    if TRAIN_AUGMENTATION:
        return transforms.Compose(
            [
                transforms.RandomResizedCrop(
                    config.IMG_SIZE,
                    scale=(0.75, 1.0),
                ),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(config.IMG_MEAN, config.IMG_STD),
            ]
        )

    return transforms.Compose(
        [
            transforms.Resize(config.IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(config.IMG_MEAN, config.IMG_STD),
        ]
    )


def create_eval_transform():
    return transforms.Compose(
        [
            transforms.Resize(config.IMG_SIZE),
            transforms.ToTensor(),
            transforms.Normalize(config.IMG_MEAN, config.IMG_STD),
        ]
    )


TRAIN_TRANSFORM = create_train_transform()
EVAL_TRANSFORM = create_eval_transform()


def make_continual_loader(
    split,
    task_ids,
    transform,
    *,
    shuffle=False,
    batch_size=BATCH_SIZE,
):
    dataset = create_continual_dataset(
        split,
        task_ids,
        image_root=config.DATA_RAW_ROOT,
        transform=transform,
    )
    loader = create_dataloader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    return dataset, loader


def save_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2)


def load_json(path: Path):
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def write_csv(path: Path, rows, fieldnames) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def get_git_commit() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=PROJECT_ROOT,
            text=True,
        ).strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"


def validate_run_task_plan(run_config: dict, run_name: str) -> None:
    if int(run_config.get("continual_num_classes", -1)) != NUM_CLASSES:
        raise ValueError(f"{run_name}: continual_num_classes does not match 100")
    if int(run_config.get("classes_per_task", -1)) != CLASSES_PER_TASK:
        raise ValueError(f"{run_name}: classes_per_task does not match 10")

    saved_sha = run_config.get("task_plan_sha256")
    if saved_sha is None:
        warnings.warn(
            f"{run_name}: run_config.json has no task_plan_sha256; "
            "the task order cannot be cryptographically verified."
        )
    elif saved_sha != TASK_PLAN_SHA256:
        raise ValueError(
            f"{run_name}: task-plan SHA-256 differs from the current shared plan. "
            "Do not mix this run with the current experiments."
        )


set_seed()


## Step 5 — Run D's existing scratch no-replay and replay entry points

This cell does not reimplement D's scratch trainers. It calls the committed scripts directly and writes versioned outputs under `outputs/continual_100/`.

In [ ]:
def is_completed_sequential_run(output_dir: Path) -> bool:
    config_path = output_dir / "run_config.json"
    if not config_path.is_file():
        return False

    run_config = load_json(config_path)
    return int(run_config.get("completed_task_id", -1)) == NUM_TASKS - 1


def run_existing_scratch_script(
    *,
    script_name: str,
    output_dir: Path,
    memory_per_class=None,
) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)

    if is_completed_sequential_run(output_dir):
        print(f"Skipping completed run: {output_dir}")
        return

    command = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / script_name),
        "--image-root",
        str(config.DATA_RAW_ROOT),
        "--output-dir",
        str(output_dir),
        "--epochs-per-task",
        str(EPOCHS_PER_TASK),
        "--batch-size",
        str(BATCH_SIZE),
        "--learning-rate",
        str(SCRATCH_LEARNING_RATE),
        "--num-workers",
        str(NUM_WORKERS),
    ]

    if TRAIN_AUGMENTATION:
        command.append("--train-augmentation")

    if memory_per_class is not None:
        command.extend(["--memory-per-class", str(memory_per_class)])

    checkpoint_path = output_dir / "last_checkpoint.pt"
    if RESUME_EXISTING_RUNS and checkpoint_path.is_file():
        command.append("--resume")

    print("Running:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)


if RUN_SCRATCH_NO_REPLAY:
    run_existing_scratch_script(
        script_name="train_continual_no_replay.py",
        output_dir=RUN_DIRECTORIES["scratch_no_replay"],
    )

if RUN_SCRATCH_REPLAY_M2:
    run_existing_scratch_script(
        script_name="train_continual_replay.py",
        output_dir=RUN_DIRECTORIES["scratch_replay_m2"],
        memory_per_class=2,
    )

if RUN_SCRATCH_REPLAY_M5:
    run_existing_scratch_script(
        script_name="train_continual_replay.py",
        output_dir=RUN_DIRECTORIES["scratch_replay_m5"],
        memory_per_class=5,
    )


## Step 6 — Model factories and optimiser settings

In [ ]:
def create_pretrained_resnet18(num_classes: int = NUM_CLASSES):
    model = resnet18(weights=ResNet18_Weights.DEFAULT)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model


def create_experiment_model(pretrained: bool):
    if pretrained:
        return create_pretrained_resnet18(NUM_CLASSES)
    return create_scratch_resnet18(NUM_CLASSES)


def create_experiment_optimizer(model, pretrained: bool):
    if pretrained:
        return optim.AdamW(
            model.parameters(),
            lr=PRETRAINED_LEARNING_RATE,
            weight_decay=WEIGHT_DECAY,
        )

    return optim.SGD(
        model.parameters(),
        lr=SCRATCH_LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )


def create_experiment_criterion(pretrained: bool):
    label_smoothing = PRETRAINED_LABEL_SMOOTHING if pretrained else 0.0
    return nn.CrossEntropyLoss(label_smoothing=label_smoothing)


## Step 7 — Joint-training upper bound

Joint training sees all ten tasks together. It is not a continual-learning method; it is an upper-bound reference with the same number of current-data image presentations as the sequential runs.

The best checkpoint is selected using the complete 100-class validation split only.

In [ ]:
JOINT_HISTORY_FIELDS = [
    "epoch",
    "train_loss",
    "train_top1",
    "val_loss",
    "val_top1",
    "epoch_seconds",
]


def create_joint_run_config(pretrained: bool):
    return {
        "approach": "joint_training_upper_bound",
        "batch_size": BATCH_SIZE,
        "classes_per_task": CLASSES_PER_TASK,
        "continual_num_classes": NUM_CLASSES,
        "epochs": JOINT_EPOCHS,
        "git_commit": get_git_commit(),
        "image_mean": config.IMG_MEAN,
        "image_root": str(config.DATA_RAW_ROOT),
        "image_size": list(config.IMG_SIZE),
        "image_std": config.IMG_STD,
        "label_smoothing": (
            PRETRAINED_LABEL_SMOOTHING if pretrained else 0.0
        ),
        "learning_rate": (
            PRETRAINED_LEARNING_RATE if pretrained else SCRATCH_LEARNING_RATE
        ),
        "model_name": (
            "pretrained_resnet18" if pretrained else "scratch_resnet18"
        ),
        "num_workers": NUM_WORKERS,
        "optimizer": "AdamW" if pretrained else "SGD",
        "pretrained": pretrained,
        "random_seed": config.RANDOM_SEED,
        "selection_split": "validation",
        "task_plan": str(TASK_PLAN_PATH),
        "task_plan_sha256": TASK_PLAN_SHA256,
        "train_augmentation": TRAIN_AUGMENTATION,
        "weight_decay": WEIGHT_DECAY,
    }


def run_joint_training(run_key: str, pretrained: bool) -> None:
    output_dir = RUN_DIRECTORIES[run_key]
    output_dir.mkdir(parents=True, exist_ok=True)

    set_seed()
    run_config = create_joint_run_config(pretrained)
    model = create_experiment_model(pretrained).to(DEVICE)
    optimizer = create_experiment_optimizer(model, pretrained)
    criterion = create_experiment_criterion(pretrained)

    _, train_loader = make_continual_loader(
        "train",
        TASK_IDS,
        TRAIN_TRANSFORM,
        shuffle=True,
    )
    _, val_loader = make_continual_loader(
        "val",
        TASK_IDS,
        EVAL_TRANSFORM,
    )

    best_path = output_dir / "best_model.pt"
    last_path = output_dir / "last_checkpoint.pt"
    history_path = output_dir / "training_history.csv"

    history_rows = []
    start_epoch = 1
    best_val_top1 = float("-inf")

    if RESUME_EXISTING_RUNS and last_path.is_file():
        checkpoint = torch.load(last_path, map_location=DEVICE, weights_only=False)
        validate_run_task_plan(checkpoint["run_config"], run_key)
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        history_rows = checkpoint["history_rows"]
        best_val_top1 = float(checkpoint["best_val_top1"])
        start_epoch = int(checkpoint["completed_epoch"]) + 1

    if start_epoch > JOINT_EPOCHS and best_path.is_file():
        print(f"Joint run is already complete: {output_dir}")
    else:
        for epoch in range(start_epoch, JOINT_EPOCHS + 1):
            epoch_start = time.perf_counter()
            train_metrics = train_one_epoch(
                model,
                train_loader,
                optimizer,
                DEVICE,
                criterion,
            )
            val_metrics = validate_one_epoch(
                model,
                val_loader,
                DEVICE,
                criterion,
            )

            history_rows.append(
                {
                    "epoch": epoch,
                    "train_loss": train_metrics["loss"],
                    "train_top1": train_metrics["top1"],
                    "val_loss": val_metrics["loss"],
                    "val_top1": val_metrics["top1"],
                    "epoch_seconds": time.perf_counter() - epoch_start,
                }
            )

            if val_metrics["top1"] > best_val_top1:
                best_val_top1 = val_metrics["top1"]
                torch.save(
                    {
                        "checkpoint_type": "joint_best",
                        "completed_epoch": epoch,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "best_val_top1": best_val_top1,
                        "run_config": run_config,
                    },
                    best_path,
                )

            run_config["completed_epoch"] = epoch
            save_json(output_dir / "run_config.json", run_config)
            write_csv(history_path, history_rows, JOINT_HISTORY_FIELDS)

            torch.save(
                {
                    "checkpoint_type": "joint_last",
                    "completed_epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "history_rows": history_rows,
                    "best_val_top1": best_val_top1,
                    "run_config": run_config,
                },
                last_path,
            )

            print(
                f"[{run_key}] epoch {epoch:02d}/{JOINT_EPOCHS} | "
                f"train={train_metrics['top1']:.4f} | "
                f"val={val_metrics['top1']:.4f}"
            )

    if not best_path.is_file():
        raise FileNotFoundError(f"Best joint checkpoint not found: {best_path}")

    best_checkpoint = torch.load(
        best_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(best_checkpoint["model_state_dict"])

    overall_val_metrics = validate_one_epoch(
        model,
        val_loader,
        DEVICE,
        criterion,
    )

    task_accuracy_rows = []
    for task_id in TASK_IDS:
        _, task_loader = make_continual_loader(
            "val",
            task_id,
            EVAL_TRANSFORM,
        )
        task_metrics = validate_one_epoch(
            model,
            task_loader,
            DEVICE,
            criterion,
        )
        task_accuracy_rows.append(
            {
                "task_id": task_id,
                "validation_accuracy": task_metrics["top1"],
            }
        )

    validation_summary = {
        "split": "validation",
        "best_validation_top1": float(best_checkpoint["best_val_top1"]),
        "overall_validation_top1": float(overall_val_metrics["top1"]),
        "overall_validation_loss": float(overall_val_metrics["loss"]),
        "mean_task_validation_accuracy": float(
            np.mean(
                [row["validation_accuracy"] for row in task_accuracy_rows]
            )
        ),
    }

    save_json(output_dir / "validation_metrics.json", validation_summary)
    write_csv(
        output_dir / "joint_task_accuracies.csv",
        task_accuracy_rows,
        ["task_id", "validation_accuracy"],
    )

    print(json.dumps(validation_summary, indent=2))


if RUN_SCRATCH_JOINT:
    run_joint_training("scratch_joint", pretrained=False)

if RUN_PRETRAINED_JOINT:
    run_joint_training("pretrained_joint", pretrained=True)


## Step 8 — Unified pretrained sequential CL trainer

This is one pretrained trainer parameterised by `memory_per_class`.

- `memory_per_class=0`: pretrained no replay;
- `memory_per_class=2`: pretrained replay m=2;
- `memory_per_class=5`: pretrained replay m=5.

It reuses the shared task map, shared Dataset/DataLoader, shared CL metrics, and shared replay-memory functions. It does not duplicate D's scratch entry points.

In [ ]:
SEQUENTIAL_HISTORY_FIELDS = [
    "task_id",
    "epoch_in_task",
    "global_epoch",
    "train_loss",
    "train_top1",
    "current_val_loss",
    "current_val_top1",
    "epoch_seconds",
]

TASK_METRIC_FIELDS = [
    "task_id",
    "current_task_accuracy",
    "old_task_accuracy",
    "seen_task_accuracy",
    "average_forgetting",
]

MEMORY_SUMMARY_FIELDS = [
    "task_id",
    "memory_per_class",
    "memory_classes",
    "memory_samples",
]


def create_pretrained_sequential_config(memory_per_class: int):
    return {
        "approach": (
            "pretrained_sequential_no_replay"
            if memory_per_class == 0
            else "pretrained_class_balanced_replay"
        ),
        "batch_size": BATCH_SIZE,
        "classes_per_task": CLASSES_PER_TASK,
        "continual_num_classes": NUM_CLASSES,
        "epochs_per_task": EPOCHS_PER_TASK,
        "git_commit": get_git_commit(),
        "image_mean": config.IMG_MEAN,
        "image_root": str(config.DATA_RAW_ROOT),
        "image_size": list(config.IMG_SIZE),
        "image_std": config.IMG_STD,
        "label_smoothing": PRETRAINED_LABEL_SMOOTHING,
        "learning_rate": PRETRAINED_LEARNING_RATE,
        "memory_per_class": memory_per_class,
        "model_name": "pretrained_resnet18",
        "num_workers": NUM_WORKERS,
        "optimizer": "AdamW",
        "pretrained": True,
        "random_seed": config.RANDOM_SEED,
        "selection_split": "validation",
        "task_plan": str(TASK_PLAN_PATH),
        "task_plan_sha256": TASK_PLAN_SHA256,
        "train_augmentation": TRAIN_AUGMENTATION,
        "weight_decay": WEIGHT_DECAY,
    }


def save_pretrained_sequential_artifacts(
    output_dir,
    run_config,
    matrix,
    history_rows,
    summary_rows,
    memory_rows,
    memory_samples,
):
    save_json(output_dir / "run_config.json", run_config)
    save_json(
        output_dir / "accuracy_matrix.json",
        {"split": "validation", "matrix": matrix},
    )
    write_csv(
        output_dir / "training_history.csv",
        history_rows,
        SEQUENTIAL_HISTORY_FIELDS,
    )
    write_csv(
        output_dir / "task_metrics.csv",
        summary_rows,
        TASK_METRIC_FIELDS,
    )

    if int(run_config["memory_per_class"]) > 0:
        write_csv(
            output_dir / "memory_summary.csv",
            memory_rows,
            MEMORY_SUMMARY_FIELDS,
        )
        write_csv(
            output_dir / "memory_samples.csv",
            [
                {
                    "file_path": relative_path,
                    "continual_label": label,
                }
                for relative_path, label in memory_samples
            ],
            ["file_path", "continual_label"],
        )


def run_pretrained_sequential(
    run_key: str,
    *,
    memory_per_class: int,
) -> None:
    if memory_per_class < 0:
        raise ValueError("memory_per_class cannot be negative")

    output_dir = RUN_DIRECTORIES[run_key]
    output_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = output_dir / "last_checkpoint.pt"

    set_seed()
    run_config = create_pretrained_sequential_config(memory_per_class)
    model = create_pretrained_resnet18(NUM_CLASSES).to(DEVICE)
    optimizer = create_experiment_optimizer(model, pretrained=True)
    criterion = create_experiment_criterion(pretrained=True)

    matrix = create_accuracy_matrix(NUM_TASKS)
    history_rows = []
    summary_rows = []
    memory_rows = []
    memory_samples = []
    start_task_id = 0
    global_epoch = 0

    if RESUME_EXISTING_RUNS and checkpoint_path.is_file():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )
        validate_run_task_plan(checkpoint["run_config"], run_key)

        saved_memory = int(
            checkpoint["run_config"].get("memory_per_class", -1)
        )
        if saved_memory != memory_per_class:
            raise ValueError(
                f"{run_key}: saved memory_per_class={saved_memory}, "
                f"requested {memory_per_class}"
            )

        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        matrix = checkpoint["accuracy_matrix"]
        history_rows = checkpoint["history_rows"]
        summary_rows = checkpoint["summary_rows"]
        memory_rows = checkpoint["memory_rows"]
        memory_samples = checkpoint["memory_samples"]
        start_task_id = int(checkpoint["completed_task_id"]) + 1
        global_epoch = len(history_rows)

    if start_task_id >= NUM_TASKS:
        print(f"Pretrained sequential run is already complete: {output_dir}")
        return

    for task_id in range(start_task_id, NUM_TASKS):
        current_dataset = create_continual_dataset(
            "train",
            task_id,
            image_root=config.DATA_RAW_ROOT,
            transform=TRAIN_TRANSFORM,
        )
        current_samples = list(current_dataset.samples)

        if memory_per_class > 0:
            current_dataset.samples = [
                *current_samples,
                *memory_samples,
            ]

        train_loader = create_dataloader(
            current_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True,
            num_workers=NUM_WORKERS,
            pin_memory=torch.cuda.is_available(),
        )

        _, current_val_loader = make_continual_loader(
            "val",
            task_id,
            EVAL_TRANSFORM,
        )

        for epoch_in_task in range(1, EPOCHS_PER_TASK + 1):
            epoch_start = time.perf_counter()
            train_metrics = train_one_epoch(
                model,
                train_loader,
                optimizer,
                DEVICE,
                criterion,
            )
            current_val_metrics = validate_one_epoch(
                model,
                current_val_loader,
                DEVICE,
                criterion,
            )
            global_epoch += 1

            history_rows.append(
                {
                    "task_id": task_id,
                    "epoch_in_task": epoch_in_task,
                    "global_epoch": global_epoch,
                    "train_loss": train_metrics["loss"],
                    "train_top1": train_metrics["top1"],
                    "current_val_loss": current_val_metrics["loss"],
                    "current_val_top1": current_val_metrics["top1"],
                    "epoch_seconds": time.perf_counter() - epoch_start,
                }
            )

            print(
                f"[{run_key}] task {task_id + 1:02d}/{NUM_TASKS} | "
                f"epoch {epoch_in_task:02d}/{EPOCHS_PER_TASK} | "
                f"train={train_metrics['top1']:.4f} | "
                f"current-val={current_val_metrics['top1']:.4f}"
            )

        task_accuracies = {}
        for evaluated_task_id in range(task_id + 1):
            _, validation_loader = make_continual_loader(
                "val",
                evaluated_task_id,
                EVAL_TRANSFORM,
            )
            task_accuracies[evaluated_task_id] = validate_one_epoch(
                model,
                validation_loader,
                DEVICE,
                criterion,
            )["top1"]

        record_accuracy_row(matrix, task_id, task_accuracies)
        summary_rows.append(summarize_after_task(matrix, task_id))

        if memory_per_class > 0:
            memory_samples = update_class_balanced_memory(
                memory_samples,
                current_samples,
                memory_per_class,
                config.RANDOM_SEED,
            )
            seen_labels = range(
                (task_id + 1) * CLASSES_PER_TASK
            )
            validate_class_balanced_memory(
                memory_samples,
                memory_per_class,
                seen_labels,
            )
            memory_rows.append(
                {
                    "task_id": task_id,
                    "memory_per_class": memory_per_class,
                    "memory_classes": len(seen_labels),
                    "memory_samples": len(memory_samples),
                }
            )

        run_config["completed_task_id"] = task_id
        run_config["epochs_completed"] = global_epoch

        save_pretrained_sequential_artifacts(
            output_dir,
            run_config,
            matrix,
            history_rows,
            summary_rows,
            memory_rows,
            memory_samples,
        )

        torch.save(
            {
                "checkpoint_type": "pretrained_continual_task_boundary_last",
                "completed_task_id": task_id,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "run_config": run_config,
                "accuracy_matrix": matrix,
                "history_rows": history_rows,
                "summary_rows": summary_rows,
                "memory_rows": memory_rows,
                "memory_samples": memory_samples,
            },
            checkpoint_path,
        )

        print(json.dumps(summary_rows[-1], indent=2))

    print(f"Saved pretrained continual outputs to: {output_dir}")


if RUN_PRETRAINED_NO_REPLAY:
    run_pretrained_sequential(
        "pretrained_no_replay",
        memory_per_class=0,
    )

if RUN_PRETRAINED_REPLAY_M2:
    run_pretrained_sequential(
        "pretrained_replay_m2",
        memory_per_class=2,
    )

if RUN_PRETRAINED_REPLAY_M5:
    run_pretrained_sequential(
        "pretrained_replay_m5",
        memory_per_class=5,
    )


## Step 9 — Final 100-class test evaluation

Use this step only after the validation comparison is complete and `TEST_RUN_KEYS` contains the final selected run or runs.

The function calls:

```python
evaluate_class_scores(targets, scores, num_classes=100)
```

The test results are written back into each selected run directory and are not used by any training or hyperparameter-selection function.

In [ ]:
def create_model_from_run_config(run_config: dict):
    model_name = run_config.get("model_name")

    if model_name == "pretrained_resnet18":
        return create_pretrained_resnet18(NUM_CLASSES)
    if model_name == "scratch_resnet18":
        return create_scratch_resnet18(NUM_CLASSES)

    raise ValueError(f"Unsupported model_name: {model_name}")


def find_run_checkpoint(run_dir: Path, run_config: dict) -> Path:
    if run_config.get("approach") == "joint_training_upper_bound":
        checkpoint_path = run_dir / "best_model.pt"
    else:
        checkpoint_path = run_dir / "last_checkpoint.pt"

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"Checkpoint not found for final test: {checkpoint_path}"
        )
    return checkpoint_path


@torch.inference_mode()
def collect_class_scores(model, loader):
    targets = []
    scores = []

    model.eval()
    for inputs, batch_targets in loader:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)

        targets.append(batch_targets.cpu().numpy())
        scores.append(outputs.cpu().numpy())

    return np.concatenate(targets), np.concatenate(scores)


def evaluate_run_on_final_test(run_key: str) -> None:
    run_dir = RUN_DIRECTORIES[run_key]
    run_config_path = run_dir / "run_config.json"

    if not run_config_path.is_file():
        raise FileNotFoundError(
            f"run_config.json not found for {run_key}: {run_config_path}"
        )

    run_config = load_json(run_config_path)
    validate_run_task_plan(run_config, run_key)

    model = create_model_from_run_config(run_config).to(DEVICE)
    checkpoint_path = find_run_checkpoint(run_dir, run_config)
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    model.load_state_dict(checkpoint["model_state_dict"])

    test_dataset, test_loader = make_continual_loader(
        "test",
        TASK_IDS,
        EVAL_TRANSFORM,
        shuffle=False,
    )

    targets, scores = collect_class_scores(model, test_loader)
    evaluation = evaluate_class_scores(
        targets,
        scores,
        num_classes=NUM_CLASSES,
    )

    save_json(
        run_dir / "test_metrics.json",
        {
            "split": "test",
            **evaluation["metrics"],
        },
    )
    np.save(
        run_dir / "test_confusion_matrix.npy",
        evaluation["confusion_matrix"],
    )
    pd.DataFrame(evaluation["confusion_matrix"]).to_csv(
        run_dir / "test_confusion_matrix.csv",
        index=False,
    )

    predictions = evaluation["predictions"]
    prediction_rows = []
    for (relative_path, target), prediction in zip(
        test_dataset.samples,
        predictions,
    ):
        prediction_rows.append(
            {
                "file_path": relative_path,
                "target": int(target),
                "prediction": int(prediction),
                "correct": int(target) == int(prediction),
            }
        )

    write_csv(
        run_dir / "test_predictions.csv",
        prediction_rows,
        ["file_path", "target", "prediction", "correct"],
    )

    print(f"Final test completed for {run_key}")
    print(json.dumps(evaluation["metrics"], indent=2))


if RUN_FINAL_TEST:
    if not TEST_RUN_KEYS:
        raise ValueError(
            "Set TEST_RUN_KEYS only after validation-based model selection."
        )

    for selected_run_key in TEST_RUN_KEYS:
        if selected_run_key not in RUN_DIRECTORIES:
            raise KeyError(f"Unknown TEST_RUN_KEYS entry: {selected_run_key}")
        evaluate_run_on_final_test(selected_run_key)


## Step 10 — Optional representation-drift analysis

This analysis is disabled by default. It does not retrain the model. It compares the ImageNet-initialised `avgpool` representation with the representation from an already trained pretrained checkpoint on a fixed, seeded subset of validation images.

In [ ]:
@torch.inference_mode()
def extract_avgpool_features(model, loader):
    batches = []

    def hook(_module, _inputs, output):
        batches.append(output.flatten(1).detach().cpu())

    handle = model.avgpool.register_forward_hook(hook)
    model.eval()

    try:
        for inputs, _targets in loader:
            model(inputs.to(DEVICE))
    finally:
        handle.remove()

    return torch.cat(batches, dim=0)


def run_representation_drift(run_key: str) -> None:
    run_dir = RUN_DIRECTORIES[run_key]
    run_config = load_json(run_dir / "run_config.json")
    validate_run_task_plan(run_config, run_key)

    if run_config.get("model_name") != "pretrained_resnet18":
        raise ValueError("Representation drift requires a pretrained ResNet-18 run")

    trained_model = create_pretrained_resnet18(NUM_CLASSES).to(DEVICE)
    checkpoint_path = find_run_checkpoint(run_dir, run_config)
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )
    trained_model.load_state_dict(checkpoint["model_state_dict"])

    initial_model = create_pretrained_resnet18(NUM_CLASSES).to(DEVICE)

    validation_dataset = create_continual_dataset(
        "val",
        TASK_IDS,
        image_root=config.DATA_RAW_ROOT,
        transform=EVAL_TRANSFORM,
    )

    sample_count = min(DRIFT_MAX_SAMPLES, len(validation_dataset))
    rng = np.random.default_rng(config.RANDOM_SEED)
    selected_indices = np.sort(
        rng.choice(
            len(validation_dataset),
            size=sample_count,
            replace=False,
        )
    )
    subset = Subset(validation_dataset, selected_indices.tolist())
    subset_loader = create_dataloader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    initial_features = extract_avgpool_features(
        initial_model,
        subset_loader,
    )
    trained_features = extract_avgpool_features(
        trained_model,
        subset_loader,
    )

    cosine_similarity = F.cosine_similarity(
        initial_features,
        trained_features,
        dim=1,
    ).numpy()

    drift_rows = []
    for subset_position, dataset_index in enumerate(selected_indices):
        relative_path, target = validation_dataset.samples[int(dataset_index)]
        drift_rows.append(
            {
                "dataset_index": int(dataset_index),
                "file_path": relative_path,
                "target": int(target),
                "cosine_similarity": float(
                    cosine_similarity[subset_position]
                ),
                "cosine_distance": float(
                    1.0 - cosine_similarity[subset_position]
                ),
            }
        )

    drift_summary = {
        "split": "validation",
        "num_samples": sample_count,
        "mean_cosine_similarity": float(cosine_similarity.mean()),
        "std_cosine_similarity": float(cosine_similarity.std()),
        "mean_cosine_distance": float(
            (1.0 - cosine_similarity).mean()
        ),
        "random_seed": config.RANDOM_SEED,
    }

    save_json(run_dir / "representation_drift_summary.json", drift_summary)
    write_csv(
        run_dir / "representation_drift_samples.csv",
        drift_rows,
        [
            "dataset_index",
            "file_path",
            "target",
            "cosine_similarity",
            "cosine_distance",
        ],
    )

    print(json.dumps(drift_summary, indent=2))


if RUN_DRIFT_ANALYSIS:
    run_representation_drift(DRIFT_RUN_KEY)


## Step 11 — Step 13 automatic run summariser

The summariser receives run directories as parameters and reads files directly. It never depends on manually copied values or renamed result files.

For sequential runs it requires:

- `run_config.json`
- `accuracy_matrix.json`
- `task_metrics.csv`
- `memory_summary.csv` when replay is enabled

For joint upper bounds it reads:

- `run_config.json`
- `validation_metrics.json`
- `joint_task_accuracies.csv`

If `test_metrics.json` exists, it is included as a final reporting field but is not used to rank or select validation configurations.

In [ ]:
matrix_path = RUN_DIRECTORIES["scratch_no_replay"] / "accuracy_matrix.json"
matrix_payload = load_json(matrix_path)

print(type(matrix_payload))
if isinstance(matrix_payload, dict):
    print("keys:", matrix_payload.keys())
    print("split:", matrix_payload.get("split"))
else:
    print("Legacy matrix without split field")

In [ ]:
from pathlib import Path
from IPython.display import display

COMPARISON_OUTPUT_DIR = OUTPUT_ROOT / "comparison_v1"


def load_optional_json(path: Path):
    """Load a JSON file when it exists; otherwise return None."""
    return load_json(path) if path.is_file() else None


def to_float_or_nan(value):
    """Convert a scalar to float while preserving missing values as NaN."""
    if value is None:
        return np.nan

    try:
        if pd.isna(value):
            return np.nan
    except TypeError:
        pass

    return float(value)


def to_int_or_default(value, default=0):
    """Convert a scalar to int, using a default for missing values."""
    if value is None:
        return default

    try:
        if pd.isna(value):
            return default
    except TypeError:
        pass

    return int(value)


def read_validation_accuracy_matrix(
    matrix_path: Path,
    run_key: str,
):
    """
    Read both supported continual-learning accuracy-matrix formats.

    Supported formats:

    1. {"split": "val", "matrix": [...]}
    2. {"split": "validation", "matrix": [...]}
    3. Legacy format where the JSON file directly contains the matrix.
    """
    matrix_payload = load_json(matrix_path)

    if isinstance(matrix_payload, dict):
        matrix_split = matrix_payload.get("split")
        accuracy_matrix = matrix_payload.get("matrix")

    elif isinstance(matrix_payload, list):
        matrix_split = None
        accuracy_matrix = matrix_payload

    else:
        raise ValueError(
            f"{run_key}: unsupported accuracy_matrix.json format "
            f"({type(matrix_payload).__name__})"
        )

    if matrix_split is not None:
        normalised_split = str(matrix_split).strip().lower()

        if normalised_split not in {"val", "validation"}:
            raise ValueError(
                f"{run_key}: accuracy_matrix.json uses unexpected "
                f"split '{matrix_split}'"
            )

    if accuracy_matrix is None:
        raise ValueError(
            f"{run_key}: accuracy_matrix.json does not contain "
            "a matrix"
        )

    if not isinstance(accuracy_matrix, list):
        raise ValueError(
            f"{run_key}: accuracy matrix must be a list"
        )

    if not accuracy_matrix:
        raise ValueError(
            f"{run_key}: accuracy matrix is empty"
        )

    return accuracy_matrix


def load_run_summary(
    run_key: str,
    run_dir: Path,
):
    """Read one run and return its comparison row and task table."""

    run_config_path = run_dir / "run_config.json"

    if not run_config_path.is_file():
        return None, None

    run_config = load_json(run_config_path)

    validate_run_task_plan(
        run_config,
        run_key,
    )

    approach = str(
        run_config.get("approach", "unknown")
    )

    model_name = run_config.get(
        "model_name",
        "unknown",
    )

    memory_per_class = to_int_or_default(
        run_config.get("memory_per_class"),
        default=0,
    )

    pretrained_value = run_config.get("pretrained")

    if pretrained_value is None:
        pretrained_value = (
            model_name == "pretrained_resnet18"
        )

    summary = {
        "run_key": run_key,
        "approach": approach,
        "model_name": model_name,
        "pretrained": bool(pretrained_value),
        "memory_per_class": memory_per_class,
        "epochs_per_task": run_config.get(
            "epochs_per_task"
        ),
        "joint_epochs": run_config.get("epochs"),
        "learning_rate": run_config.get(
            "learning_rate"
        ),
        "validation_top1": np.nan,
        "final_current_task_accuracy": np.nan,
        "final_old_task_accuracy": np.nan,
        "final_seen_task_accuracy": np.nan,
        "final_average_forgetting": np.nan,
        "memory_samples": 0,
        "test_top1": np.nan,
        "test_top5": np.nan,
        "test_macro_f1": np.nan,
    }

    task_metrics_table = None

    is_joint_run = (
        approach == "joint_training_upper_bound"
    )

    # Joint-training upper bound
    if is_joint_run:
        validation_path = (
            run_dir / "validation_metrics.json"
        )

        task_accuracy_path = (
            run_dir / "joint_task_accuracies.csv"
        )

        if not validation_path.is_file():
            raise FileNotFoundError(
                f"{run_key}: missing "
                f"{validation_path.name}"
            )

        if not task_accuracy_path.is_file():
            raise FileNotFoundError(
                f"{run_key}: missing "
                f"{task_accuracy_path.name}"
            )

        validation_metrics = load_json(
            validation_path
        )

        validation_top1 = validation_metrics.get(
            "overall_validation_top1",
            validation_metrics.get(
                "best_validation_top1"
            ),
        )

        if validation_top1 is None:
            raise ValueError(
                f"{run_key}: validation_metrics.json "
                "contains no validation Top-1 value"
            )

        summary["validation_top1"] = float(
            validation_top1
        )

    # Sequential continual-learning runs
    else:
        matrix_path = (
            run_dir / "accuracy_matrix.json"
        )

        task_metrics_path = (
            run_dir / "task_metrics.csv"
        )

        if not matrix_path.is_file():
            raise FileNotFoundError(
                f"{run_key}: missing "
                f"{matrix_path.name}"
            )

        if not task_metrics_path.is_file():
            raise FileNotFoundError(
                f"{run_key}: missing "
                f"{task_metrics_path.name}"
            )

        # Accepts both "val" and "validation".
        read_validation_accuracy_matrix(
            matrix_path,
            run_key,
        )

        task_metrics_table = pd.read_csv(
            task_metrics_path
        )

        if task_metrics_table.empty:
            raise ValueError(
                f"{run_key}: task_metrics.csv is empty"
            )

        required_metric_columns = {
            "task_id",
            "current_task_accuracy",
            "old_task_accuracy",
            "seen_task_accuracy",
            "average_forgetting",
        }

        missing_metric_columns = (
            required_metric_columns.difference(
                task_metrics_table.columns
            )
        )

        if missing_metric_columns:
            raise ValueError(
                f"{run_key}: task_metrics.csv is "
                "missing columns: "
                + ", ".join(
                    sorted(missing_metric_columns)
                )
            )

        for column in required_metric_columns:
            task_metrics_table[column] = (
                pd.to_numeric(
                    task_metrics_table[column],
                    errors="coerce",
                )
            )

        task_metrics_table = (
            task_metrics_table
            .sort_values("task_id")
            .reset_index(drop=True)
        )

        final_row = task_metrics_table.iloc[-1]

        summary["validation_top1"] = (
            to_float_or_nan(
                final_row[
                    "seen_task_accuracy"
                ]
            )
        )

        summary[
            "final_current_task_accuracy"
        ] = to_float_or_nan(
            final_row[
                "current_task_accuracy"
            ]
        )

        summary[
            "final_old_task_accuracy"
        ] = to_float_or_nan(
            final_row[
                "old_task_accuracy"
            ]
        )

        summary[
            "final_seen_task_accuracy"
        ] = to_float_or_nan(
            final_row[
                "seen_task_accuracy"
            ]
        )

        summary[
            "final_average_forgetting"
        ] = to_float_or_nan(
            final_row[
                "average_forgetting"
            ]
        )

        if memory_per_class > 0:
            memory_path = (
                run_dir / "memory_summary.csv"
            )

            if not memory_path.is_file():
                raise FileNotFoundError(
                    f"{run_key}: replay run is "
                    "missing memory_summary.csv"
                )

            memory_table = pd.read_csv(
                memory_path
            )

            if memory_table.empty:
                raise ValueError(
                    f"{run_key}: "
                    "memory_summary.csv is empty"
                )

            if (
                "memory_samples"
                not in memory_table.columns
            ):
                raise ValueError(
                    f"{run_key}: "
                    "memory_summary.csv has no "
                    "memory_samples column"
                )

            summary["memory_samples"] = (
                to_int_or_default(
                    memory_table.iloc[-1][
                        "memory_samples"
                    ],
                    default=0,
                )
            )

    # Optional final test results
    test_metrics = load_optional_json(
        run_dir / "test_metrics.json"
    )

    if test_metrics is not None:
        summary["test_top1"] = (
            to_float_or_nan(
                test_metrics.get("top1")
            )
        )

        summary["test_top5"] = (
            to_float_or_nan(
                test_metrics.get("top5")
            )
        )

        summary["test_macro_f1"] = (
            to_float_or_nan(
                test_metrics.get("macro_f1")
            )
        )

    return summary, task_metrics_table


def build_step13_summary(
    run_directories: dict,
):
    """
    Read all available run directories, create one
    comparison table, and save the comparison figures.
    """

    COMPARISON_OUTPUT_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    summaries = []
    sequential_tables = {}

    for run_key, run_dir in (
        run_directories.items()
    ):
        run_dir = Path(run_dir)

        try:
            summary, task_table = (
                load_run_summary(
                    run_key,
                    run_dir,
                )
            )

        except Exception as error:
            print(
                f"Skipping {run_key}: "
                f"{type(error).__name__}: "
                f"{error}"
            )
            continue

        if summary is None:
            print(
                f"Skipping {run_key}: "
                "run_config.json not found in "
                f"{run_dir}"
            )
            continue

        summaries.append(summary)

        if task_table is not None:
            sequential_tables[
                run_key
            ] = task_table

        print(f"Loaded: {run_key}")

    if not summaries:
        print(
            "No usable completed runs were found. "
            "Check the folders under "
            "outputs/continual_100."
        )

        return pd.DataFrame()

    summary_table = pd.DataFrame(
        summaries
    )

    summary_table = (
        summary_table
        .sort_values(
            [
                "validation_top1",
                "run_key",
            ],
            ascending=[
                False,
                True,
            ],
            na_position="last",
        )
        .reset_index(drop=True)
    )

    summary_csv_path = (
        COMPARISON_OUTPUT_DIR
        / "comparison_summary.csv"
    )

    summary_table.to_csv(
        summary_csv_path,
        index=False,
    )

    display(summary_table)

    # Validation comparison bar chart
    validation_plot = (
        summary_table[
            [
                "run_key",
                "validation_top1",
            ]
        ]
        .dropna(
            subset=["validation_top1"]
        )
    )

    if not validation_plot.empty:
        figure = plt.figure(
            figsize=(11, 5)
        )

        axis = figure.add_subplot(111)

        axis.bar(
            validation_plot["run_key"],
            validation_plot[
                "validation_top1"
            ],
        )

        axis.set_title(
            "Validation performance by run"
        )

        axis.set_xlabel("Run")

        axis.set_ylabel(
            "Validation Top-1 / "
            "final seen-task accuracy"
        )

        axis.tick_params(
            axis="x",
            rotation=45,
        )

        figure.tight_layout()

        figure.savefig(
            COMPARISON_OUTPUT_DIR
            / "validation_comparison.png",
            dpi=200,
        )

        plt.show()

    # Sequential seen-task accuracy curves
    if sequential_tables:
        figure = plt.figure(
            figsize=(10, 5)
        )

        axis = figure.add_subplot(111)

        for run_key, table in (
            sequential_tables.items()
        ):
            axis.plot(
                table["task_id"] + 1,
                table[
                    "seen_task_accuracy"
                ],
                marker="o",
                label=run_key,
            )

        axis.set_title(
            "Seen-task validation accuracy "
            "after each task"
        )

        axis.set_xlabel("Completed task")

        axis.set_ylabel(
            "Seen-task accuracy"
        )

        axis.set_xticks(
            range(
                1,
                NUM_TASKS + 1,
            )
        )

        axis.legend()

        figure.tight_layout()

        figure.savefig(
            COMPARISON_OUTPUT_DIR
            / "seen_accuracy_curves.png",
            dpi=200,
        )

        plt.show()

        # Forgetting curves
        figure = plt.figure(
            figsize=(10, 5)
        )

        axis = figure.add_subplot(111)

        for run_key, table in (
            sequential_tables.items()
        ):
            valid_rows = table.dropna(
                subset=[
                    "average_forgetting"
                ]
            )

            if valid_rows.empty:
                continue

            axis.plot(
                valid_rows["task_id"] + 1,
                valid_rows[
                    "average_forgetting"
                ],
                marker="o",
                label=run_key,
            )

        axis.set_title(
            "Average forgetting "
            "after each task"
        )

        axis.set_xlabel("Completed task")

        axis.set_ylabel(
            "Average forgetting"
        )

        axis.set_xticks(
            range(
                2,
                NUM_TASKS + 1,
            )
        )

        axis.legend()

        figure.tight_layout()

        figure.savefig(
            COMPARISON_OUTPUT_DIR
            / "forgetting_curves.png",
            dpi=200,
        )

        plt.show()

    # Replay-memory comparison
    replay_table = (
        summary_table[
            summary_table[
                "memory_per_class"
            ] > 0
        ]
        .dropna(
            subset=[
                "final_seen_task_accuracy"
            ]
        )
    )

    if not replay_table.empty:
        figure = plt.figure(
            figsize=(8, 5)
        )

        axis = figure.add_subplot(111)

        for model_name, group in (
            replay_table.groupby(
                "model_name",
                dropna=False,
            )
        ):
            group = group.sort_values(
                "memory_per_class"
            )

            axis.plot(
                group[
                    "memory_per_class"
                ],
                group[
                    "final_seen_task_accuracy"
                ],
                marker="o",
                label=str(model_name),
            )

        axis.set_title(
            "Replay budget versus "
            "final seen-task accuracy"
        )

        axis.set_xlabel(
            "Memory samples per class"
        )

        axis.set_ylabel(
            "Final seen-task accuracy"
        )

        axis.legend()

        figure.tight_layout()

        figure.savefig(
            COMPARISON_OUTPUT_DIR
            / "memory_budget_comparison.png",
            dpi=200,
        )

        plt.show()

    print(
        "Comparison table saved to:",
        summary_csv_path,
    )

    print(
        "Figures saved to:",
        COMPARISON_OUTPUT_DIR,
    )

    return summary_table


# Step 11 only reads saved results.
# It does not retrain any model.
comparison_summary = build_step13_summary(
    RUN_DIRECTORIES
)

## Step 12 — Report-use checklist

Before using any number in the group report:

1. confirm that `run_config.json` records the shared task-plan SHA-256;
2. confirm that the run used 100 classes and 10 classes per task;
3. confirm that `accuracy_matrix.json` is marked as validation during configuration selection;
4. compare the validation runs first;
5. freeze the selected learning rate, replay budget, epoch count, and model choice;
6. run the held-out test only once for the selected configuration;
7. report joint training as an upper bound, not as a continual-learning method;
8. report replay memory in both samples per class and total stored samples;
9. include final seen-task accuracy and average forgetting;
10. keep representation drift as an optional analysis rather than a selection criterion.